# 6장 심화 실습 — 경로 해석과 도달 범위 분석

6장의 기본 구현을 바탕으로 출발시각과 도달 범위를 비교합니다. 교재 6.7절의 선택 활동입니다.
이 노트북은 제공된 RAPTOR로 하남 자료를 처음부터 읽습니다.
출발시각 비교, 도달 범위 지도, 시간대별 분석을 기본 구현과 분리해 실행합니다.

아래 셀에서 패키지 경로와 한글 글꼴을 설정합니다.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

import pandas as pd
import matplotlib.pyplot as plt
from lab import expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

제공된 시간표·경로 함수와 시각 표시 함수를 읽습니다. 이 노트북은 학생 파일의 완성 여부와 관계없이 실행됩니다.

In [ ]:
from smartmob.data import load_gtfs, parse_gtfs_time, seconds_to_gtfs_time
from smartmob.teaching.raptor import (
    INF, TransitData, toy_feed, raptor as ref_raptor, journey, summarize,
)

def clock_label(seconds):
    return "미도달" if seconds == INF else seconds_to_gtfs_time(seconds)

이 사례를 통과하면 해당 조건에서 손으로 계산한 답과 일치합니다.
다른 시간표의 정확성까지 증명된 것은 아닙니다.

## 1. GTFS 자료 변환과 경로 탐색 (교재 6.6)

이 절부터는 제공된 구현을 사용합니다. 자료 준비는 한 번 하고 질의에서 재사용합니다.

In [ ]:
from time import perf_counter

hanam = load_gtfs("hanam")
started = perf_counter()
data = TransitData.from_gtfs(hanam)
print("준비 시간(초):", round(perf_counter() - started, 2))
print(data.describe())
print("출발순이 뒤집힌 패턴:", sum(not p._sorted for p in data.patterns))

패턴 349개, 운행 8,923개, 방향을 구분한 도보 연결 28,818개입니다.
출발순이 뒤집힌 패턴도 있어 제공된 함수는 해당 목록을 직접 훑습니다.
이 처리만으로 추월하는 운행의 최선 도착이 모두 보장되지는 않습니다.

In [ ]:
origin = (37.5393, 127.2148)
destination = (37.5606, 127.1930)
origins = data.access_stops(*origin)
target = data.nearest_stop(*destination)
departure = 8 * 3600
started = perf_counter()
result = ref_raptor(data, origins, departure)
query_ms = (perf_counter() - started) * 1000
print("접근 정류장:", len(origins), "도착 정류장:", data.stop_names[target])
print("유한한 도착시각:", sum(t < INF for t in result.best), "개")
print("질의 시간(ms):", round(query_ms, 1))

접근 후보는 최대 30개이며 도착 정류장은 ‘미사강변브라운스톤’입니다.
4,156개 정류장에 도착시각이 나옵니다. 30분 도달 수가 아니라 시간표 전체와 탑승 다섯 번의 결과입니다.
질의 시간은 컴퓨터마다 달라지므로 고정 정답으로 쓰지 않습니다.

## 2. 경로 복원과 통행시간 구성 (교재 6.6)

`journey`로 복원한 구간에 정류장 이름을 붙입니다.
도보 구간과 운행의 승하차시각을 함께 읽습니다.

In [ ]:
legs = journey(data, result, target)
for leg in legs:
    if leg["kind"] == "transit":
        print(leg["route"], data.stop_names[leg["from"]], "→", data.stop_names[leg["to"]],
              clock_label(leg["board_time"]), "→", clock_label(leg["alight_time"]))
    else:
        print("도보", leg["seconds"], "초 →", data.stop_names[leg["to"]])
summary = summarize(data, legs, departure)
summary

약 22.9분 중 차내 6.6분, 도보 11.2분, 대기 5.1분입니다.
마지막 정류장에서 목적지 좌표까지의 도보는 빠져 있습니다.
출발을 5분 늦춰 같은 시간표에서 선택한 운행과 시간 내역을 비교합니다.

In [ ]:
comparisons = []
for start in [departure, departure + 300]:
    res = ref_raptor(data, origins, start)
    parts = journey(data, res, target)
    comparisons.append({"출발": clock_label(start), "도착": clock_label(res.best[target]),
                        **summarize(data, parts, start)})
pd.DataFrame(comparisons)

출발 차이와 도착 차이를 비교합니다. 같은 차를 탈 수 있으면 대기가 줄고,
차를 놓치면 다음 운행의 도착시각을 사용합니다. 시각표에 없는 지연은 반영하지 않습니다.

## 3. 경로 탐색 결과의 불변식 검증 (교재 6.7)

제공된 결과에서 출발 전 도착, 출발을 늦췄을 때의 더 이른 도착,
탑승 횟수를 늘렸을 때의 더 늦은 도착이 있는지 검사합니다. 미도달도 무한대로 비교합니다.

In [ ]:
later = ref_raptor(data, origins, departure + 1800)
assert all(t >= departure for t in result.best)
assert all(a <= b for a, b in zip(result.best, later.best))
for prev, cur in zip(result.rounds, result.rounds[1:]):
    assert all(b <= a for a, b in zip(prev, cur))
print("세 불변식에서 위반을 찾지 못했습니다.")

세 조건에 위반이 없더라도 모든 경로가 정확하다는 증명은 아닙니다. 다음으로 허용 탑승 횟수와 시간 한도를 바꾸어 봅니다.

## 4. 탑승 횟수 제한과 도달 범위 (교재 6.7)

제공된 `result.rounds`에서 유한한 정류장 수와 30분 이내 수를 함께 셉니다.
라운드 수는 탑승 횟수입니다. 도보로 도달한 정류장도 포함합니다.

In [ ]:
coverage = pd.DataFrame([
    {"최대 탑승 횟수": k, "시간표 안에서 도달": sum(t < INF for t in row),
     "30분 이내": sum(t <= departure + 1800 for t in row)}
    for k, row in enumerate(result.rounds)
]).set_index("최대 탑승 횟수")
display(coverage)
coverage.plot(marker="o", figsize=(8, 3), ylabel="도달 정류장 수")
plt.tight_layout()

탑승을 더 허용해도 이전 답은 남습니다. 시간표 전체에서 도달하는 수와 30분 이내 수의 차이를 읽습니다.

### 출발시각·탑승 상한에 따른 도달 범위 비교

제공된 구현으로 하남 GTFS를 계산한 뒤 HTML에서 결과를 비교합니다.
통행시간 30분을 고정하고 탑승 상한을 1회에서 2회로 늘려 봅니다.
‘작은 시간표로 따라가기’에서는 Python 함수의 계산 순서도 단계별로 재생합니다.

In [ ]:
from smartmob.viz.transit import raptor_explorer

raptor_explorer(hanam, view="hanam")

지도의 정류장을 누르면 라운드별 도착시각이 나옵니다. 오른쪽 경로는 본문 목적지까지의 결과입니다.
03:00 출발은 첫차를 기다릴 수 있지만 30분 도달 범위는 작습니다.
이 화면은 제공된 구현을 사용하며 학생 코드의 채점을 대신하지 않습니다.

이제 30·45·60분 범위를 정류장 점으로 그립니다. 배경 지도 다운로드는 필요하지 않습니다.

In [ ]:
import numpy as np

minutes = (np.array(result.best) - departure) / 60
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(data.stop_lons, data.stop_lats, s=3, color="#d8dfe7", label="실습 정류장")
for lower, upper, color in [(45, 60, "#8560a9"), (30, 45, "#168579"), (-1, 30, "#2869a4")]:
    mask = (minutes > lower) & (minutes <= upper)
    ax.scatter(np.array(data.stop_lons)[mask], np.array(data.stop_lats)[mask],
               s=9, color=color, label=f"{max(lower, 0)}~{upper}분")
ax.scatter(origin[1], origin[0], marker="*", s=150, color="#d0801d", label="하남시청 출발")
ax.set(xlabel="경도", ylabel="위도", title="08:00 출발 · 최대 5번 탑승 · 도달 정류장")
ax.set_aspect(1 / np.cos(np.radians(origin[0])))
ax.legend()
fig.tight_layout()

각 색은 출발 뒤 도보·대기·차내시간을 모두 더한 구간입니다.
점을 이어 면으로 채우지 않았으므로 점 사이의 모든 장소에 같은 시간 안에 갈 수 있다는 뜻은 아닙니다.

## 5. 연습 — 출발시각에 따른 도달 범위 변화 ★★

07시부터 23시까지 한 시간 간격으로 같은 목적지의 통행시간을 구해 봅시다.
자료 `data`와 접근 후보 `origins`를 재사용합니다. 미도달은 `None`으로 기록해 선이 끊기게 합니다.

산출물: 시간대별 그래프 1장, 시간이 크게 다른 두 시각의 경로와 대기시간 비교 3문장.
아래 `hourly`에 `출발시`, `통행시간(분)` 두 컬럼의 표를 넣습니다.

In [ ]:
hourly = None

todo("시간대별 통행시간 표", hourly)
if hourly is not None:
    hourly.plot(x="출발시", y="통행시간(분)", marker="o", figsize=(8, 3))
    plt.tight_layout()

미도달을 0분으로 넣으면 가장 빠른 경로처럼 보이므로 따로 표시합니다.
추가 연습 ★★★: 출발 좌표를 바꾼 뒤 접근 후보도 다시 만들고 30·45·60분 도달 지도를 비교해 봅시다.
산출물은 지도 2장과 시간 한도별 정류장 수 표입니다. 정류장 수를 인구 수로 해석하지 않습니다.